In [ ]:
import numpy as np
import joblib
from sklearn.decomposition import IncrementalPCA
from sklearn.cluster import MiniBatchKMeans

In [ ]:
data= np.load("user_means_norm_total_and_counts.npz")
user_means = data["user_means"].item()

In [ ]:
ipca = joblib.load("pca_model.joblib")

In [ ]:
names = list(user_means.keys())
batch_size = ipca.batch_size

In [ ]:
for i in range(0, len(names), batch_size):
    batch_names = names[i:i+batch_size]

    # Batch bauen
    X_batch = np.stack([user_means[n] for n in batch_names]).astype(np.float32)

    # PCA transform
    X_batch_pca = ipca.transform(X_batch)

    # Inplace überschreiben
    for j, name in enumerate(batch_names):
        user_means[name] = X_batch_pca[j]

    del X_batch, X_batch_pca

In [ ]:
mbk = joblib.load("pca_model.joblib")

In [ ]:
embeddings_raw = np.stack(list(user_means.values()))
usernames = list(user_means.keys())

In [ ]:
pca_dim= 20

In [ ]:
embeddings = embeddings_raw[:, :pca_dim]

In [ ]:
cluster_ids = mbk.predict(embeddings)

In [ ]:
user_cluster_dict = dict(zip(usernames, cluster_ids))

In [ ]:
np.savez_compressed(
    "user_cluster_dict_total.npz",
    user_cluster_dict=user_cluster_dict
)